In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_classic.chains.question_answering.map_rerank_prompt import output_parser
from langchain_classic.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

LLM chain

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [21]:
llm = ChatOllama(model=os.getenv('MODEL_NAME'), temperature=os.getenv("TEMPERATURE"))

In [22]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
    "Give me only ONE name. No list"
)

In [23]:
chain = LLMChain(
    llm=llm,
    prompt=prompt,
)

In [24]:
product = "Queen Size Sheet Set"
chain.invoke(product)

{'product': 'Queen Size Sheet Set', 'text': 'RoyalRest'}

Sequential Chains

In [25]:
### 顺序链是按预定义顺序执行其链接的链
### SimpleSequentialChain：是最简单的，每个步骤都有一个输入输出，前一个步骤的输出是下一个步骤的输入
from langchain_classic.chains import SimpleSequentialChain

In [26]:
llm = ChatOllama(model=os.getenv('MODEL_NAME'), temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
    "Give me only ONE name. No list"
)

# chain 1
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
)

In [27]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company:{company_name}"
)

# chain 2
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
)

In [28]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [29]:
overall_simple_chain.invoke(product)



> Entering new SimpleSequentialChain chain...
RoyalSize Sheets
RoyalSize Sheets offers luxurious, extra-large bedding for a premium sleep experience in spacious bedrooms.

> Finished chain.


{'input': 'Queen Size Sheet Set',
 'output': 'RoyalSize Sheets offers luxurious, extra-large bedding for a premium sleep experience in spacious bedrooms.'}

In [30]:
### SequentialChain：有多个输入或多个输出，链中的输入和输出keys要正确对齐
from langchain_classic.chains import SequentialChain

In [31]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="English_Review"
                    )

In [32]:
# prompt template 2: summarize the review
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summary"
                    )

In [33]:
# prompt template 3: review language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

In [34]:
# prompt template 4: follow-up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

In [40]:
# overall_chain: 指定完整链的最终输入和输出：input= Review
# and output= English_Review, summary, language, followup_message
# 此处的output和input与中间变量key无关，只决定最终输出的内容有哪些key
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    # output_variables=["English_Review", "summary", "language", "followup_message"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)

In [41]:
df = pd.read_csv("data/Data.csv")
print(df.head(), "\n")

review = df.Review[5]
overall_chain.invoke(review)

                   Product                                             Review
0     Queen Size Sheet Set  I ordered a king size set. My only criticism w...
1   Waterproof Phone Pouch  I loved the waterproof sac, although the openi...
2      Luxury Air Mattress  This mattress had a small hole in the top of i...
3           Pillows Insert  This is the best throw pillow fillers on Amazo...
4  Milk Frother Handheld\n   I loved this product. But they only seem to l... 



> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': 'Here\'s the English translation of the review:\n\n"I find the taste mediocre. The foam doesn\'t last, it\'s strange. I buy the same ones in stores and the taste is much better...\nOld batch or counterfeit?! "',
 'summary': "The reviewer finds the product's taste mediocre, with poor lasting foam, and suspects it might be an old batch or counterfeit due to a significant difference in quality compared to store-bought versions.",
 'followup_message': "Réponse suivante en français :\n\nCher(e) [Nom du destinataire],\n\nJe voulais vous faire part de mon expérience avec ce produit que j'ai récemment acheté. Comme indiqué dans ma courte réflexion, je suis déçu par la qualité globale du produit. \n\nD'abord, le goût est à mes yeux insipide et ne correspond pas aux attentes établies par les produits s

Router Chain

In [52]:
### 路由链，针对输入的类型，决定应该调用哪一条子链
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate


# 定义适合于不同场景下的prompt template
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question: {input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts,
answer the component parts, and then put them together to answer the broader question.

Here is a question: {input}"""


history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people, events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and evaluate the past. You have a respect for historical evidence and the ability to make use of it to support your explanations and judgements.

Here is a question: {input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration, forward-thinking, confidence, strong problem-solving capabilities, understanding of theories and algorithms, and excellent communication skills. \
You are great at answering coding questions. \
You are so good because you know how to solve a problem by describing the solution in imperative steps that a machine can easily interpret and you know how to choose a solution that has a good balance between time complexity and space complexity.

Here is a question: {input}"""

In [46]:
# 为每个prompt template 进行命名和描述
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics.",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering questions about mathematics.",
        "prompt_template": math_template
    },
    {
        "name": "history",
        "description": "Good for answering questions about history.",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering questions about computer science.",
        "prompt_template": computerscience_template
    }
]

In [47]:
# destination_chains：基于prompt template 创建对应的目标链
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [
    f"{p['name']}: {p['description']}" for p in prompt_infos
]
destination_str = "\n".join(destinations)

In [48]:
# default_chain：当router无法决定使用哪个子链的时候，调用默认目标链
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [50]:
# 定义不同链之间的路由模板，包含要执行的任务说明，以及输出应具备的特定格式
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a description of what the prompt is best suited for. \
You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [62]:
# router_chain：根据定义的router template创建router prompt，并创建路由链
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destination_str)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(), # 帮助chains决定在哪条子链之间进行路由
)

router_chain = LLMRouterChain.from_llm(
    llm=ChatOllama(model=os.getenv('MODEL_NAME'), temperature=0),
    prompt=router_prompt)

In [64]:
# 最后，创建整体链路
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True
)

In [68]:
chain.invoke("what's black body radiation?")



> Entering new MultiPromptChain chain...
None: {'input': "What's black body radiation?"}
> Finished chain.


{'input': "What's black body radiation?",
 'text': 'Blackbody radiation is a concept from physics that describes the electromagnetic radiation emitted by a perfect absorber and re-emitter of radiation (a so-called "blackbody") at thermal equilibrium. Here’s a more detailed explanation:\n\n1. **Definition**: A blackbody is an idealized physical body that absorbs all incident electromagnetic radiation, regardless of frequency or angle of incidence.\n\n2. **Radiation Spectrum**: According to the theory of quantum mechanics and thermodynamics, any blackbody will emit electromagnetic radiation at all frequencies (or wavelengths) with a spectrum that depends only on its temperature. The intensity of this radiation follows a specific distribution known as the Planck\'s law.\n\n3. **Planck’s Law**: This law describes the spectral density of electromagnetic radiation emitted by a black body in thermal equilibrium at a given temperature \\( T \\). Mathematically, it is expressed as:\n\n   \\[\n 

In [67]:
chain.invoke("what's 1+1?")



> Entering new MultiPromptChain chain...
math: {'input': "What's 1+1?"}
> Finished chain.


{'input': "What's 1+1?",
 'text': 'Sure! The problem 1 + 1 is a simple addition problem. When we add 1 and 1 together, we get:\n\n\\[ 1 + 1 = 2 \\]\n\nThis is one of the basic facts in arithmetic, often learned early on as part of fundamental mathematics education. If you need further explanation or want to explore more about this concept, feel free to ask!'}

In [65]:
chain.invoke("why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'why does every cell in our body contain DNA?'}
> Finished chain.


{'input': 'why does every cell in our body contain DNA?',
 'text': "Every cell in the human body contains DNA because DNA (deoxyribonucleic acid) is essential for storing and transmitting genetic information from one generation of cells to the next. Here’s why:\n\n1. **Genetic Information**: DNA carries the instructions necessary for constructing, maintaining, and reproducing every part of a living organism. These instructions are encoded in genes, which dictate traits such as eye color, blood type, and susceptibility to certain diseases.\n\n2. **Cell Division (Mitosis)**: When cells divide—through a process known as mitosis—they need to replicate their genetic material so that each new cell receives an exact copy of the original DNA. This ensures that all parts of the body maintain consistency in structure and function.\n\n3. **Development and Growth**: During fetal development and throughout life, cells must multiply to support growth, repair tissues, replace worn-out or dead cells, 